# VSM Cultural Alignment Probe (Hofstede VSM) — Interactive Notebook

Reads `vsm_questions.csv` and probes any HuggingFace instruction-tuned model.
Produces a CSV compatible with `calculate_idv.py` and `calculate_nvas_hf.py`.

This is the notebook version of `vsm_probe.py` — the same functions, split into cells
by module so each stage (device detection, model loading, prompting, inference, the
full batch run) can be run and re-run independently while the model stays loaded in memory.

**Framing conditions** (Zahraei & Asgari, 2025):
- `neutral`  — no cultural identity (control)
- `persona`  — "Imagine you are an average Indonesian person"
- `observer` — "How would an average Indonesian respond to..."

**Suggested workflow:** run cells top to bottom once. After the model is loaded, you can
re-run the "Interactive single-question test" cell as many times as you like — tweak the
question, framing, or scale and re-run without reloading the model — before committing to
the full batch run at the bottom.

## 1. Imports & Setup

In [ ]:
import csv
import json
import os
import re
import time
import importlib.util
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer

load_dotenv()  # loads HF_TOKEN from .env into os.environ
warnings.filterwarnings("ignore", category=UserWarning)

_BNB_AVAILABLE = importlib.util.find_spec("bitsandbytes") is not None

## 2. Configuration

Edit values here and re-run this cell before loading the model. If you change
`model_id` or `use_4bit_quantization` after the model is already loaded, re-run the
**Model Loader** section below to pick up the change.

In [ ]:
CONFIG = {
    # ── Model ───────────────────────────────────────
    "model_id" : "Yellow-AI-NLP/komodo-7b-base",
    # 4-bit quantization (GPU only, requires bitsandbytes)
    "use_4bit_quantization": True,

    # HF token — required for gated models (Llama, Mistral)
    "hf_token": os.getenv("HF_TOKEN") or None,

    # ── Input / Output ──────────────────────────
    "questions_file": "vsm_questions.csv",

    # ── Experimental design ─────────────────────
    # Runs per question for reliability (Khan et al. FAccT'25 use 3; Hadar-Shoval use 10)
    "runs_per_question": 10,

    # Framing conditions — remove any you don't need
    "framing_conditions": ["neutral", "persona", "observer"],

    # Target culture for persona / observer framings
    "target_culture": "Indonesian",

    # ── Generation ───────────────────────────
    "temperature"     : 0.0,   # 0 = greedy/deterministic
    "max_new_tokens"  : 5,

    # ── Performance ───────────────────────────
    "delay_between_calls": 0.1,
    "cache_dir": None,
}
CONFIG

## 3. Device Detection

In [ ]:
def detect_device() -> str:
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU: {name} ({mem:.1f} GB VRAM)")
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("Apple Silicon MPS detected")
        return "mps"
    print("No GPU — running on CPU (slow; consider a smaller model)")
    return "cpu"

In [ ]:
device = detect_device()
device

## 4. Model Loader

This downloads the model on first run — may take a while for larger models.

In [ ]:
def load_model_and_tokenizer(config: dict, device: str):
    model_id = config["model_id"]
    print(f"\nLoading: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=config["hf_token"],
        cache_dir=config["cache_dir"],
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs = {
        "token"           : config["hf_token"],
        "cache_dir"       : config["cache_dir"],
        "trust_remote_code": True,
    }

    if config["use_4bit_quantization"] and device == "cuda" and _BNB_AVAILABLE:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        # Keep all layers on GPU — CPU offload conflicts with 4-bit bitsandbytes
        gpu_mem = torch.cuda.get_device_properties(0).total_memory
        reserved_mb = 512
        kwargs["max_memory"] = {0: f"{(gpu_mem // 1024**2) - reserved_mb}MiB"}
        kwargs["device_map"] = "auto"
        kwargs["offload_folder"] = "offload_cache"
        print("4-bit quantization enabled")
    elif device == "cuda":
        if config["use_4bit_quantization"] and not _BNB_AVAILABLE:
            print("bitsandbytes not installed — falling back to FP16")
            print("Install with: pip install bitsandbytes>=0.46.1")
        kwargs["dtype"]          = torch.float16
        kwargs["device_map"]     = "auto"
        kwargs["offload_folder"] = "offload_cache"
    elif device == "mps":
        kwargs["dtype"] = torch.float16
    else:
        kwargs["dtype"] = torch.float32

    remote_cfg = AutoConfig.from_pretrained(
        model_id, token=config["hf_token"], cache_dir=config["cache_dir"], trust_remote_code=True
    )
    _VL_SUFFIXES = ("vl", "vision", "multimodal")
    is_vl = any(s in type(remote_cfg).__name__.lower() for s in _VL_SUFFIXES)
    ModelCls = AutoModelForImageTextToText if is_vl else AutoModelForCausalLM
    model = ModelCls.from_pretrained(model_id, **kwargs)
    if device in ("cpu", "mps") and "device_map" not in kwargs:
        model = model.to(device)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e9
    print(f"Model loaded ({n_params:.2f}B parameters)\n")
    return model, tokenizer

In [ ]:
model, tokenizer = load_model_and_tokenizer(CONFIG, device)

## 5. Prompt Builder

In [ ]:
def build_messages(question_text: str, framing: str,
                   culture: str, scale_min: int, scale_max: int) -> list[dict]:
    scale_note = (
        f"Respond with ONLY a single integer from {scale_min} to {scale_max}. "
        f"No explanation, no punctuation — just the digit."
    )

    if framing == "neutral":
        system = "You are a person completing a survey. Answer every question with only a single digit."
        user   = f"{question_text}\n\n{scale_note}"

    elif framing == "persona":
        system = (
            f"Imagine you are an average {culture} person. "
            f"Answer every survey question from that cultural perspective "
            f"using only a single digit."
        )
        user = f"{question_text}\n\n{scale_note}"

    elif framing == "observer":
        system = "You are a cross-cultural researcher. Answer with only a single digit."
        user   = (
            f"How would an average {culture} person respond to this survey question?\n\n"
            f"{question_text}\n\n{scale_note}"
        )

    else:
        raise ValueError(f"Unknown framing: {framing!r}")

    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]

In [ ]:
# Quick sanity check — inspect the messages for one framing
build_messages(
    question_text="How important is it to have a job that leaves enough time for personal life?",
    framing="persona",
    culture=CONFIG["target_culture"],
    scale_min=1, scale_max=5,
)

## 6. Inference

In [ ]:
def extract_score(text: str, scale_min: int, scale_max: int):
    for m in re.findall(r'\b(\d+)\b', text.strip()):
        v = int(m)
        if scale_min <= v <= scale_max:
            return float(v)
    return None


_digit_token_id_cache: dict[tuple[int, int], int | None] = {}


def _digit_token_id(tokenizer, scale_min: int, scale_max: int) -> int | None:
    """Single-token id representing a digit in [scale_min, scale_max], cached
    per (scale_min, scale_max) since it's invariant across calls for a given tokenizer."""
    key = (scale_min, scale_max)
    if key not in _digit_token_id_cache:
        token_id = None
        for digit in range(scale_min, scale_max + 1):
            for token_str in [str(digit), f" {digit}"]:
                ids = tokenizer.encode(token_str, add_special_tokens=False)
                if len(ids) == 1:
                    token_id = ids[0]
                    break
            if token_id is not None:
                break
        _digit_token_id_cache[key] = token_id
    return _digit_token_id_cache[key]


def digit_logprob(scores_tuple, tokenizer, scale_min: int, scale_max: int) -> float | None:
    """
    Returns the log-probability of the first generated digit token.
    scores_tuple: model.generate outputs.scores (tuple of tensors, one per new token)
    """
    if not scores_tuple:
        return None
    token_id = _digit_token_id(tokenizer, scale_min, scale_max)
    if token_id is None:
        return None
    first_logits = scores_tuple[0]          # shape (batch, vocab)
    log_probs    = torch.log_softmax(first_logits[0], dim=-1)
    return round(log_probs[token_id].item(), 6)

In [ ]:
@torch.no_grad()
def query_model(messages: list[dict], model, tokenizer,
                config: dict, device: str,
                scale_min: int, scale_max: int) -> dict:
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        prompt = "\n".join(
            f"{m['role'].upper()}: {m['content']}" for m in messages
        ) + "\nASSISTANT:"

    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=2048
    )
    target = model.device if device == "mps" else device
    inputs = {k: v.to(target) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    gen_kwargs = {
        "max_new_tokens"          : config["max_new_tokens"],
        "do_sample"               : config["temperature"] > 0,
        "pad_token_id"            : tokenizer.pad_token_id,
        "eos_token_id"            : tokenizer.eos_token_id,
        "return_dict_in_generate" : True,
        "output_scores"           : True,
    }
    if config["temperature"] > 0:
        gen_kwargs["temperature"] = config["temperature"]

    t0      = time.time()
    outputs = model.generate(**inputs, **gen_kwargs)
    elapsed = round(time.time() - t0, 3)

    new_tokens = outputs.sequences[0][input_len:]
    raw_text   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Some models (e.g. Gemma 3) wrap digits in special tokens that get stripped.
    # Fall back to decoding without skipping, then remove angle-bracket tokens.
    if not raw_text and len(new_tokens) > 0:
        raw_full = tokenizer.decode(new_tokens, skip_special_tokens=False)
        raw_text = re.sub(r"<[^>]+>", "", raw_full).strip()

    score      = extract_score(raw_text, scale_min, scale_max)
    lp         = digit_logprob(outputs.scores, tokenizer, scale_min, scale_max)

    return {
        "raw_response"     : raw_text,
        "extracted_score"  : score,
        "logprob"          : lp,
        "input_tokens"     : input_len,
        "generation_time_s": elapsed,
    }

### Interactive single-question test

Run this cell as many times as you like — change the question, framing, or scale and
re-run without reloading the model.

In [ ]:
test_messages = build_messages(
    question_text="How important is it to have a job that leaves enough time for personal life?",
    framing="persona",
    culture=CONFIG["target_culture"],
    scale_min=1, scale_max=5,
)

result = query_model(test_messages, model, tokenizer, CONFIG, device, scale_min=1, scale_max=5)
result

## 7. Question Loader

In [ ]:
def load_questions(filepath: str) -> list[dict]:
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Questions file not found: {filepath}")

    df = pd.read_csv(filepath)
    required = {"question_id", "dimension", "question_text", "scale_min", "scale_max"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"vsm_questions.csv is missing columns: {missing}")

    df["scale_min"] = df["scale_min"].astype(int)
    df["scale_max"] = df["scale_max"].astype(int)

    questions = df.to_dict(orient="records")
    print(f"Loaded {len(questions)} questions from {filepath}")
    return questions

In [ ]:
questions = load_questions(CONFIG["questions_file"])
pd.DataFrame(questions).head()

## 8. Main Runner

In [ ]:
OUTPUT_FIELDNAMES = [
    "question_id", "run_number", "dimension", "question_text",
    "framing_condition", "prompt", "raw_response", "extracted_score",
    "temperature", "scale_min", "scale_max", "logprob",
    "input_tokens", "generation_time_s",
    "model_id", "timestamp",
]


def _print_banner() -> None:
    print("=" * 60)
    print("  VSM Cultural Alignment Probe")
    print("=" * 60)


def _print_experiment_summary(config: dict, device: str, n_questions: int,
                              total_calls: int, output_path: Path) -> None:
    print(f"\nExperiment summary:")
    print(f"  Model      : {config['model_id']}")
    print(f"  Device     : {device.upper()}")
    print(f"  Questions  : {n_questions}")
    print(f"  Framings   : {config['framing_conditions']}")
    print(f"  Runs/Q     : {config['runs_per_question']}")
    print(f"  Total calls: {total_calls}")
    print(f"  Output     : {output_path}\n")


def _build_result_row(q: dict, run: int, framing: str, messages: list[dict],
                      model, tokenizer, config: dict, device: str,
                      scale_min: int, scale_max: int) -> dict:
    """Runs one probe call and returns a CSV-ready row, capturing errors inline
    so a single failed call doesn't abort the rest of the experiment."""
    row = {
        "question_id"      : q["question_id"],
        "run_number"       : run,
        "dimension"        : q["dimension"],
        "question_text"    : q["question_text"],
        "framing_condition": framing,
        "temperature"      : config["temperature"],
        "scale_min"        : scale_min,
        "scale_max"        : scale_max,
        "model_id"         : config["model_id"],
    }
    try:
        result = query_model(
            messages, model, tokenizer,
            config, device, scale_min, scale_max
        )
        row.update({
            "prompt"           : messages,
            "raw_response"     : result["raw_response"],
            "extracted_score"  : result["extracted_score"] if result["extracted_score"] is not None else "",
            "logprob"          : result["logprob"] if result["logprob"] is not None else "",
            "input_tokens"     : result["input_tokens"],
            "generation_time_s": result["generation_time_s"],
        })
    except Exception as e:
        row.update({
            "raw_response"     : f"ERROR: {str(e)[:120]}",
            "extracted_score"  : "",
            "logprob"          : "",
            "input_tokens"     : 0,
            "generation_time_s": 0,
        })
    row["timestamp"] = datetime.now().isoformat()
    return row


def _print_summary(path: Path) -> None:
    df    = pd.read_csv(path)
    total = len(df)
    ok    = df["extracted_score"].notna().sum()
    print(f"\nExtraction success: {ok}/{total} ({ok/total*100:.1f}%)")
    if ok:
        by_framing = df.groupby("framing_condition")["extracted_score"].mean()
        print("\nMean score by framing:")
        print(by_framing.to_string())


def run_probe(config: dict) -> None:
    _print_banner()

    device_           = detect_device()
    model_, tokenizer_ = load_model_and_tokenizer(config, device_)
    questions_        = load_questions(config["questions_file"])

    total_calls = (
        len(questions_) *
        len(config["framing_conditions"]) *
        config["runs_per_question"]
    )

    slug        = config["model_id"].replace("/", "_").replace(" ", "_")
    output_path = Path(f"vsm_responses_{slug}.csv")

    _print_experiment_summary(config, device_, len(questions_), total_calls, output_path)

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDNAMES)
        writer.writeheader()

        pbar = tqdm(total=total_calls, desc="Probing", unit="call")

        for q in questions_:
            scale_min = q["scale_min"]
            scale_max = q["scale_max"]

            for framing in config["framing_conditions"]:
                messages = build_messages(
                    question_text=q["question_text"],
                    framing=framing,
                    culture=config["target_culture"],
                    scale_min=scale_min,
                    scale_max=scale_max,
                )

                for run in range(1, config["runs_per_question"] + 1):
                    row = _build_result_row(
                        q, run, framing, messages,
                        model_, tokenizer_, config, device_,
                        scale_min, scale_max,
                    )
                    writer.writerow(row)
                    f.flush()   # crash-safe: write after every response
                    pbar.update(1)
                    time.sleep(config["delay_between_calls"])

            if device_ == "cuda":
                torch.cuda.empty_cache()

        pbar.close()

    print(f"\nDone. Output saved to: {output_path}")
    _print_summary(output_path)

## 9. Run the full probe

This re-detects the device and reloads the model/questions internally (so it's
self-contained and matches `python vsm_probe.py`), then runs the full batch and writes
`vsm_responses_<model_slug>.csv`. This can take a while depending on model size and
`runs_per_question` × questions × framings.

In [ ]:
run_probe(CONFIG)